## imports

In [ ]:
from openai import OpenAI
from pathlib import Path
import json
import numpy as np
import os
from collections import defaultdict
from  crawler.features_extract_weapon_batch import create_batch_jsonl_per_task
client = OpenAI()

## Send batch

In [ ]:
def submit_selected_batches(
    batch_dir: str,
    batch_ids_file: str,
    selected_files=None,         # ["role.jsonl", "age.jsonl"] או נתיבים מלאים
    include_patterns=None,       # ["*age*.jsonl"]
    exclude_patterns=None,       # ["*old*.jsonl"]
    skip_if_already_logged=True, # לא לשלוח שוב אם כבר רשום ב-extract_batches.json
    key_field="file"             # איך מזהים כפילות בקובץ המעקב
):
    """
    Upload+submit רק קבצים נבחרים מתוך batch_dir (או נתיבים מלאים),
    ומעדכן extract_batches.json - מחליף batch_id עבור אותו קובץ במקום להוסיף שורה חדשה.
    """

    batch_dir = Path(batch_dir)

    # ---------- load existing tracking ----------
    p_track = Path(batch_ids_file)
    if p_track.exists():
        try:
            existing = json.loads(p_track.read_text(encoding="utf-8"))
            if not isinstance(existing, list):
                existing = []
        except Exception:
            existing = []
    else:
        existing = []

    # Create a dictionary for fast lookup by filename
    existing_dict = {
        row.get(key_field): row 
        for row in existing 
        if isinstance(row, dict) and row.get(key_field)
    }

    # ---------- build file list ----------
    batch_files = []

    if selected_files is not None:
        # explicit list (names or paths)
        for x in selected_files:
            px = Path(x)

            # אם זה רק שם קובץ (relative בלי תיקייה) -> נצמיד ל-batch_dir
            if not px.is_absolute() and px.parent == Path('.'):
                px = batch_dir / px.name

            if px.suffix.lower() != ".jsonl":
                # אם מישהו נתן שם בלי סיומת, אפשר לנסות להוסיף
                candidate = px.with_suffix(".jsonl")
                if candidate.exists():
                    px = candidate

            batch_files.append(px)

    else:
        # default: all jsonl in folder
        batch_files = list(batch_dir.glob("*.jsonl"))

        if include_patterns:
            included = set()
            for pat in include_patterns:
                included.update(batch_dir.glob(pat))
            batch_files = list(included)

        if exclude_patterns:
            excluded = set()
            for pat in exclude_patterns:
                excluded.update(batch_dir.glob(pat))
            batch_files = [f for f in batch_files if f not in excluded]

    batch_files = sorted(set(batch_files), key=lambda p: p.name.lower())

    if not batch_files:
        print("⚠️ No batch files selected.")
        return list(existing_dict.values())

    # ---------- submit ----------
    updated_count = 0
    added_count = 0
    
    for batch_file in batch_files:
        # existence check
        if not batch_file.exists():
            print(f"❌ Missing file: {batch_file}")
            continue

        # skip duplicates by file name
        if skip_if_already_logged and batch_file.name in existing_dict:
            print(f"↩️ Skipping (already tracked): {batch_file.name}")
            continue

        print(f"\n📤 Uploading {batch_file.name}...")

        with open(batch_file, "rb") as f:
            uploaded = client.files.create(file=f, purpose="batch")
        print(f"   File ID: {uploaded.id}")

        batch = client.batches.create(
            input_file_id=uploaded.id,
            endpoint="/v1/chat/completions",
            completion_window="24h"
        )
        print(f"   Batch ID: {batch.id} - Status: {batch.status}")

        entry = {
            "file": batch_file.name,
            "file_id": uploaded.id,
            "batch_id": batch.id
        }
        
        # Check if file already exists in tracking
        if batch_file.name in existing_dict:
            print(f"   🔄 Replacing batch_id for existing file: {batch_file.name}")
            existing_dict[batch_file.name].update(entry)
            updated_count += 1
        else:
            existing_dict[batch_file.name] = entry
            added_count += 1

    # ---------- save (with updates or additions) ----------
    merged = list(existing_dict.values())
    p_track.parent.mkdir(parents=True, exist_ok=True)
    p_track.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"\n✅ Added {added_count} new batches")
    print(f"✅ Updated {updated_count} existing batches")
    print(f"✅ Total tracked batches: {len(merged)}")
    print(f"✅ Batch IDs saved/updated to: {p_track}")

    return merged


In [ ]:
list_to_send = ['age.jsonl','all_weapon_offenses.jsonl','Criminal_record.jsonl']
batch_ids = submit_selected_batches(
    batch_dir=r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\batches",
    batch_ids_file=r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\extract_batches\extract_batches.json",
    selected_files=list_to_send,
    skip_if_already_logged=False
)

## check batch status

In [ ]:
def check_batch_status(batch_ids_file: str):
    """Check status of all submitted batches."""
    
    with open(batch_ids_file, "r") as f:
        batches = json.load(f)
    
    all_done = True
    for b in batches:
        if b['file'] not in list_to_send:
            continue
        status = client.batches.retrieve(b["batch_id"])
        print(f"{b['file']}: {status.status} ({status.request_counts.completed}/{status.request_counts.total})")
        if status.status not in ["completed", "failed", "cancelled"]:
            all_done = False
    
    return all_done
# list_to_send = ['main_sentence.jsonl']
check_batch_status(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\extract_batches\extract_batches.json")

## check errors

In [ ]:
def download_batch_errors(batch_ids_file: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    with open(batch_ids_file, "r", encoding="utf-8") as f:
        batches = json.load(f)

    for b in batches:
        batch_id = b["batch_id"]
        file_name = b.get("file", batch_id)

        status = client.batches.retrieve(batch_id)

        if status.error_file_id:
            print(f"⬇ Downloading errors for {file_name}")

            error_content = client.files.content(status.error_file_id)

            out_path = os.path.join(
                output_dir,
                f"{file_name}_errors.jsonl"
            )

            with open(out_path, "wb") as out:
                out.write(error_content.read())

        else:
            print(f"✅ No errors for {file_name}")
download_batch_errors(
    batch_ids_file=r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\extract_batches\extract_batches.json",
    output_dir=r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\batch_errors"
)


## check prices

In [ ]:
import json
from collections import defaultdict

# -------------------------
# Pricing per model ($ per 1M tokens)
# -------------------------
PRICING = {
    "gpt-4.1-mini": {
        "input": 0.2,
        "output": 0.8,
    },
    "gpt-5-mini": {
        "input": 0.125,
        "output": 1.0,
    }
}

# -------------------------
# Aggregates
# -------------------------
total_requests = 0
total_cost = 0.0

tokens_by_model = defaultdict(lambda: {
    "input": 0,
    "output": 0,
    "requests": 0,
    "cost": 0.0
})

# 🆕 Aggregates per task
tokens_by_task = defaultdict(lambda: {
    "input": 0,
    "output": 0,
    "requests": 0,
    "cost": 0.0
})

# 🆕 Aggregates per model+task combination
tokens_by_model_task = defaultdict(lambda: defaultdict(lambda: {
    "input": 0,
    "output": 0,
    "requests": 0,
    "cost": 0.0
}))

# -------------------------
# Load file
# -------------------------
with open(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\extract_batches\all_weapon.jsonl",
    encoding="utf-8"
) as f:
    for line in f:
        obj = json.loads(line)

        # Skip failed requests
        if obj.get("error") is not None:
            continue

        # 🆕 Extract task from custom_id
        custom_id = obj.get("custom_id", "")
        if "__" in custom_id:
            task_name = custom_id.split("__")[1]
        else:
            task_name = "unknown"

        body = obj["response"]["body"]
        usage = body["usage"]
        model = body.get("model", "")

        # Normalize model name (prefix match)
        if model.startswith("gpt-4.1-mini"):
            model_key = "gpt-4.1-mini"
        elif model.startswith("gpt-5-mini"):
            model_key = "gpt-5-mini"
        else:
            # Unknown model → skip or raise
            print(f"Unknown model: {model}")
            continue

        input_tokens = usage.get("prompt_tokens", 0)
        output_tokens = usage.get("completion_tokens", 0)

        pricing = PRICING[model_key]

        cost = (
            input_tokens / 1_000_000 * pricing["input"] +
            output_tokens / 1_000_000 * pricing["output"]
        )

        # Update totals
        total_requests += 1
        total_cost += cost

        # Update by model
        tokens_by_model[model_key]["input"] += input_tokens
        tokens_by_model[model_key]["output"] += output_tokens
        tokens_by_model[model_key]["requests"] += 1
        tokens_by_model[model_key]["cost"] += cost

        # 🆕 Update by task
        tokens_by_task[task_name]["input"] += input_tokens
        tokens_by_task[task_name]["output"] += output_tokens
        tokens_by_task[task_name]["requests"] += 1
        tokens_by_task[task_name]["cost"] += cost

        # 🆕 Update by model+task
        tokens_by_model_task[model_key][task_name]["input"] += input_tokens
        tokens_by_model_task[model_key][task_name]["output"] += output_tokens
        tokens_by_model_task[model_key][task_name]["requests"] += 1
        tokens_by_model_task[model_key][task_name]["cost"] += cost

# -------------------------
# Print summary
# -------------------------
print("=" * 80)
print("📊 OVERALL SUMMARY")
print("=" * 80)
print(f"Total requests: {total_requests:,}")
print(f"Estimated total cost: ${total_cost:.4f}\n")

print("=" * 80)
print("📈 BREAKDOWN BY MODEL")
print("=" * 80)
for model, stats in sorted(tokens_by_model.items()):
    print(f"\nModel: {model}")
    print(f"  Requests: {stats['requests']:,}")
    print(f"  Input tokens: {stats['input']:,}")
    print(f"  Output tokens: {stats['output']:,}")
    print(f"  Total tokens: {stats['input'] + stats['output']:,}")
    print(f"  Cost: ${stats['cost']:.4f}")

print("\n" + "=" * 80)
print("📋 BREAKDOWN BY TASK")
print("=" * 80)
for task, stats in sorted(tokens_by_task.items(), key=lambda x: x[1]['cost'], reverse=True):
    print(f"\nTask: {task}")
    print(f"  Requests: {stats['requests']:,}")
    print(f"  Input tokens: {stats['input']:,}")
    print(f"  Output tokens: {stats['output']:,}")
    print(f"  Total tokens: {stats['input'] + stats['output']:,}")
    print(f"  Cost: ${stats['cost']:.4f}")
    print(f"  Avg cost per request: ${stats['cost']/stats['requests']:.6f}")

print("\n" + "=" * 80)
print("🔍 BREAKDOWN BY MODEL + TASK")
print("=" * 80)
for model, tasks in sorted(tokens_by_model_task.items()):
    print(f"\n{'─' * 80}")
    print(f"Model: {model}")
    print(f"{'─' * 80}")
    for task, stats in sorted(tasks.items(), key=lambda x: x[1]['cost'], reverse=True):
        print(f"  Task: {task}")
        print(f"    Requests: {stats['requests']:,}")
        print(f"    Input tokens: {stats['input']:,}")
        print(f"    Output tokens: {stats['output']:,}")
        print(f"    Total tokens: {stats['input'] + stats['output']:,}")
        print(f"    Cost: ${stats['cost']:.4f}")
        print(f"    Avg cost per request: ${stats['cost']/stats['requests']:.6f}")
        print()